In [1]:
# https://www.kaggle.com/maunish/clrp-roberta-svm
# https://www.kaggle.com/anaverageengineer/comlrp-baseline-for-complete-beginners

In [2]:
import spacy
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm.notebook import tqdm
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge

In [3]:
RANDOM_STATE = 41
nlp = spacy.load('en_core_web_lg')

In [4]:
COMPETITION_DATA_PATH = Path('../input/commonlitreadabilityprize')
TRAIN_DATA_PATH = COMPETITION_DATA_PATH / 'train.csv'
TEST_DATA_PATH = COMPETITION_DATA_PATH / 'test.csv'

In [5]:
train_data = pd.read_csv(TRAIN_DATA_PATH)
test_data = pd.read_csv(TEST_DATA_PATH)
print(f'Length of training data: {len(train_data)}')
print(f'Length of test data: {len(test_data)}')

Length of training data: 2834
Length of test data: 7


# Feature Extraction from spacy
All Credits to [Sumit Kumar](https://www.kaggle.com/anaverageengineer) @anaverageengineer https://www.kaggle.com/anaverageengineer/comlrp-baseline-for-complete-beginners

In [6]:
with nlp.disable_pipes():
    X_train = np.vstack([nlp(text).vector for text in tqdm(train_data['excerpt'])])
    y_train = train_data['target']
    print(f'Shape of Train vectors: {X_train.shape}')

    X_test = np.vstack([nlp(text).vector for text in tqdm(test_data['excerpt'])])
    print(f'Shape of Test vectors: {X_test.shape}')

  0%|          | 0/2834 [00:00<?, ?it/s]

Shape of Train vectors: (2834, 300)


  0%|          | 0/7 [00:00<?, ?it/s]

Shape of Test vectors: (7, 300)


# Ridge Regressor
From my own notebook: https://www.kaggle.com/vigneshbaskaran/commonlit-a1-pure-scikit-learn

In [7]:
regressor = Ridge(fit_intercept=True, normalize=False)
scores = cross_val_score(regressor, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error')
print(f'Average Root mean squared error: {np.abs(np.mean(scores))}')

Average Root mean squared error: 0.6730600873499558


# Refit model and submit

In [8]:
regressor = regressor.fit(X_train, y_train)
test_data['target'] = regressor.predict(X_test)
test_data[['id','target']].to_csv('submission.csv', index=False)